# Phase-space trajectories at points a, b, c (Fig. 5 of the article)

Trajectories from a 13×13 grid of initial memory amplitudes ($b = 0$), at $\varepsilon_d/2g_2 = 3$ and
$g_2/2\pi = 60, 90, 105$ MHz, with $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$ and $\kappa_b = \pi/10.4$.
On the origin-start Lyapunov map (`figure_lyapunov_heatmap.ipynb`) at this drive, the Hill line is at 81 MHz and the
trajectory from the origin turns chaotic at 93.7 MHz: a is regular, b is between the two lines, c is chaotic.

Same integrator as the Lyapunov maps (flux form, VelocityVerlet, `dt = get_dt`). Blue: trajectories that stay close to the
real axis (cats), red: the others (satellites and chaotic trajectories). Figures: `figures/limit_cycles/phase_space_traj_g2_*MHz_12_10_2026.png (PNG: the PDFs were 6–8 MB each)`.

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5

##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4
κ_a = 0

ϵ_p_of(g2_MHz) = asin(g2_MHz*2π*1e-3/(E_J*φ_a^2*φ_b))   # inverse of g2 = E_J sin(ϵ_p) φ_a² φ_b (static pump, ω_p = 0)

In [ ]:
# Trajectories from a grid of initial a (b = 0), flux form + VelocityVerlet with the time step of the Lyapunov maps.
# Only the last 1000 samples (≈ 5.9 ns) are kept, in the rotating frame a_R = i a e^{iω_a t} (cats on the real axis).
t_max = 30/κ_b
t_min = 0.0
dt_save = 2pi/(30*sqrt(2)*ω_a)
t_save = t_max - 1000*dt_save:dt_save:t_max
environment = MeanField(κ_a, κ_b)
Re_a_list = range(-6, 6, 13)
Im_a_list = range(-6, 6, 13)
grid = vec(collect(Iterators.product(Re_a_list, Im_a_list)))

function run_grid(ϵ_p, ϵ_d, equation)
    drive_prepared = prepare(DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase))
    traj = Vector{Vector{ComplexF64}}(undef, length(grid))
    Threads.@threads for k in eachindex(grid)
        re_i, im_j = grid[k]
        system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, sqrt(2*Z_a)*re_i, sqrt(2/Z_a)*im_j, 0.0, 0.0)
        system_prepared = prepare(system)
        environment_prepared = prepare(system, environment)
        dt = get_dt(system, environment)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        ϕ_a, ϕ_b, q_a, q_b, t_list = default_solver(get_init(system_prepared, environment_prepared), (t_min, t_max), p, dt; equation=equation, t_save=t_save)
        a = ϕ_a/sqrt(2*Z_a) .+ 1im*q_a*sqrt(Z_a/2)
        traj[k] = a.*exp.(1im*ω_a*t_list)*1im
    end
    return Dict(grid[k] => traj[k] for k in eachindex(grid))
end

is_cat(z) = maximum(abs.(imag.(z))) <= 1.0   # cats on the real axis; satellites and chaotic trajectories leave it

function plot_grid(trajs, title)
    p0 = plot()
    for z in values(trajs)
        is_cat(z) && scatter!(p0, real.(z), imag.(z), ms = 1, msw = 0, color = RGBA(13/256,76/256,116/256,1), label = "")
    end
    for z in values(trajs)
        is_cat(z) || scatter!(p0, real.(z), imag.(z), ms = 1, msw = 0, color = RGBA(185/256,21/256,21/256,1), label = "")
    end
    plot!(p0, xlims = (-7.5, 7.5), ylims = (-7.5, 7.5), xticks = -5:5:5, yticks = -5:5:5,
          xlabel = L"\mathrm{Re}(a)", ylabel = L"\mathrm{Im}(a)", title = title,
          guidefontsize = 16, tickfontsize = 14, titlefontsize = 16,
          minorgrid = true, minorgridalpha = 0.15, minorticks = 2, top_margin = 5mm, aspect_ratio = 1, size = size_px_car)
    return p0
end

In [ ]:
points = [(60.0, 3.0), (88.0, 3.0), (105.0, 3.0)]   # (g2/2π in MHz, ε_d/2g2)
Final_a = Dict{Float64, Dict{Tuple{Float64,Float64}, Vector{ComplexF64}}}()
for (g2_MHz, α_sq) in points
    ϵ_p = ϵ_p_of(g2_MHz)
    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b
    Final_a[g2_MHz] = run_grid(ϵ_p, 2α_sq*g2, default_equation!)
    println(g2_MHz, " MHz: ", count(is_cat, values(Final_a[g2_MHz])), " cat trajectories out of ", length(grid))
end

In [ ]:
for (g2_MHz, α_sq) in points
    p0 = plot_grid(Final_a[g2_MHz], L"g_2/2\pi = " * "$(Int(g2_MHz)) MHz")
    display(p0)
    savefig(p0, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/limit_cycles/phase_space_traj_g2_$(Int(g2_MHz))MHz_12_10_2026.png")
end